# Task 3 – LongEval-USim: Next Query Prediction (18 Runs)

**Voraussetzung:** Diese 3 Dateien müssen im selben Ordner wie dieses Notebook liegen:
- `task3_longeval_usim-sessions-train.csv` ← **neuer offizieller Trainingsdatensatz (snapshot-1)**
- `task3_longeval_usim-sessions-06-08_2025.csv`
- `task3_longeval_usim-sessions-09-11_2025.csv`

> **Hinweis:** Snapshot-2 und -3 werden **nicht neu generiert** – sie werden automatisch
> aus dem bestehenden ZIP des jeweiligen Runs in `runfiles_task3/` übernommen.
> Es wird **nur snapshot-1** mit dem neuen Trainingsdatensatz neu berechnet.

---

**Drei Dimensionen:**
- `query_mode`: `first` | `all` | `last` – welche Queries der Session als Kontext
- `prompt_variant`: `A` (eigener Prompt: Topics + TF-IDF, doc_variant ändert Instruktion) | `B` (Jüri-Prompt: Queries + Docs)
- `doc_variant`: `rel` | `non_rel` | `contrastive`

**doc_variant bei Prompt A** – ändert die Instruktion ans LLM:
- `rel` → Nutzer hat relevante Docs gefunden → LLM soll **tiefere / spezifischere** Folgequeries vorhersagen
- `non_rel` → Nutzer hat nichts Passendes gefunden → LLM soll **alternative / umformulierte** Queries vorhersagen
- `contrastive` → neutrale Vorhersage

**doc_variant bei Prompt B** – ändert welche Docs im Prompt erscheinen:
- `rel` → nur angeklickte Docs
- `non_rel` → nur ignorierte Docs
- `contrastive` → beide

| # | query_mode | prompt | doc_variant | run_name |
|---|-----------|--------|-------------|----------|
| 1  | `first` | `A` | `rel`         | `openai_first_A_rel` |
| 2  | `first` | `A` | `non_rel`     | `openai_first_A_non_rel` |
| 3  | `first` | `A` | `contrastive` | `openai_first_A_contrastive` |
| 4  | `first` | `B` | `rel`         | `openai_first_B_rel` |
| 5  | `first` | `B` | `non_rel`     | `openai_first_B_non_rel` |
| 6  | `first` | `B` | `contrastive` | `openai_first_B_contrastive` |
| 7  | `all`   | `A` | `rel`         | `openai_all_A_rel` |
| 8  | `all`   | `A` | `non_rel`     | `openai_all_A_non_rel` |
| 9  | `all`   | `A` | `contrastive` | `openai_all_A_contrastive` |
| 10 | `all`   | `B` | `rel`         | `openai_all_B_rel` |
| 11 | `all`   | `B` | `non_rel`     | `openai_all_B_non_rel` |
| 12 | `all`   | `B` | `contrastive` | `openai_all_B_contrastive` |
| 13 | `last`  | `A` | `rel`         | `openai_last_A_rel` |
| 14 | `last`  | `A` | `non_rel`     | `openai_last_A_non_rel` |
| 15 | `last`  | `A` | `contrastive` | `openai_last_A_contrastive` |
| 16 | `last`  | `B` | `rel`         | `openai_last_B_rel` |
| 17 | `last`  | `B` | `non_rel`     | `openai_last_B_non_rel` |
| 18 | `last`  | `B` | `contrastive` | `openai_last_B_contrastive` |


In [1]:
%pip install openai python-dotenv --quiet
%pip install git+https://github.com/clef-longeval/ir-datasets-longeval --quiet

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, time, re, zipfile, ast
import pandas as pd
from pathlib import Path
from collections import defaultdict
from dotenv import load_dotenv
from openai import OpenAI

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / '.env').exists():
        load_dotenv(folder / '.env', override=True)
        break

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '').strip().strip('"').strip("'")
if not OPENAI_API_KEY:
    raise ValueError('OPENAI_API_KEY fehlt!')
client = OpenAI(api_key=OPENAI_API_KEY)
print('OpenAI Client bereit.')

OpenAI Client bereit.


In [3]:
# !! NUR DIESE 4 WERTE ÄNDERN für jeden Run !!
RUN_CONFIG = {
    'query_mode':     'first',               # 'first' | 'last' | 'all'
    'prompt_variant': 'A',                   # 'A' = eigener Prompt | 'B' = Jüri-Prompt
    'doc_variant':    'non_rel',                 # 'rel' | 'non_rel' | 'contrastive'
    'run_name':       'openai_first_A_non_rel',  # Namen aus der Tabelle oben übernehmen
    # --- ab hier nichts ändern ---
    'model':          'gpt-4o-mini',
    'team_name':      'JOINorDIE',
    'description':    'LLM-based next query prediction using OpenAI GPT-4o-mini.',
    'max_doc_chars':  600,
    'n_rel_docs':     3,
    'n_nonrel_docs':  3,
    'output_dir':     'runfiles_task3',
}

# Session-CSVs – snapshot-1 = neuer offizieller Trainingsdatensatz
# Snapshot-2 und -3 werden NICHT neu generiert (aus bestehendem ZIP übernommen)
SNAPSHOT_FILES = {
    'snapshot-1.jsonl': 'task3_longeval_usim-sessions-train.csv',
    'snapshot-2.jsonl': 'task3_longeval_usim-sessions-06-08_2025.csv',
    'snapshot-3.jsonl': 'task3_longeval_usim-sessions-09-11_2025.csv',
}

# Prüfen ob snapshot-1 vorhanden ist (snapshot-2/3 werden nicht neu geladen)
snap1_csv = SNAPSHOT_FILES['snapshot-1.jsonl']
if not Path(snap1_csv).exists():
    print(f'FEHLER – Datei fehlt: {snap1_csv}')
    print('Bitte task3_longeval_usim-sessions-train.csv in diesen Ordner kopieren.')
else:
    os.makedirs(RUN_CONFIG['output_dir'], exist_ok=True)
    print(f"Run: {RUN_CONFIG['run_name']}")
    print(f"  query_mode={RUN_CONFIG['query_mode']} | prompt={RUN_CONFIG['prompt_variant']} | doc_variant={RUN_CONFIG['doc_variant']}")
    print(f'  Snapshot-1 CSV gefunden: {snap1_csv} ✓')
    # Prüfe ob existierender ZIP vorhanden (für snapshot-2/3)
    existing_zip = Path(RUN_CONFIG['output_dir']) / f"task3_submission_{RUN_CONFIG['run_name']}.zip"
    if existing_zip.exists():
        print(f'  Bestehender ZIP gefunden: {existing_zip.name} ✓ (snapshot-2 & -3 werden daraus übernommen)')
    else:
        print(f'  WARNUNG: Kein bestehender ZIP für diesen Run gefunden.')
        print(f'  Snapshot-2 und -3 werden ebenfalls neu generiert (dauert länger + kostet mehr).')


In [4]:
# Dokument-Lookup – nur für Prompt B nötig
doc_lookup = {}

if RUN_CONFIG['prompt_variant'] == 'B':
    from ir_datasets_longeval import load as ld_load
    print('Lade Dokument-Lookup (~5-10 Min, einmalig)...')
    dataset = ld_load('longeval-sci-2026/snapshot-1/train')
    for i, doc in enumerate(dataset.docs_iter()):
        doc_lookup[str(doc.doc_id)] = {
            'title':    getattr(doc, 'title', '') or '',
            'abstract': getattr(doc, 'abstract', '') or ''
        }
        if i % 300000 == 0 and i > 0:
            print(f'  {i:,} Docs geladen...')
    print(f'Fertig: {len(doc_lookup):,} Docs.')
else:
    print('Prompt A: kein Doc-Lookup nötig.')
    print('Topics werden on-the-fly aus den Session-Queries generiert (Abgabe-Ansatz).')

Prompt A: kein Doc-Lookup nötig.
Topics werden on-the-fly aus den Session-Queries generiert (Abgabe-Ansatz).


In [5]:
def parse_docnos(val):
    try:
        return [str(x) for x in ast.literal_eval(str(val))]
    except:
        return []

def parse_interactions(val):
    try:
        items = ast.literal_eval(str(val))
        return [str(item[0]) for item in items if isinstance(item, (list, tuple))]
    except:
        return []

def load_sessions(csv_path):
    df = pd.read_csv(csv_path, header=None)
    df.columns = ['idx','user','session_id','query','timestamp',
                  'retrieved_docs','session_hash','interactions']
    sessions = defaultdict(lambda: {'queries': [], 'rel': set(), 'nonrel': set()})
    for _, row in df.sort_values('timestamp').iterrows():
        sid = str(row['session_id'])
        sessions[sid]['queries'].append(str(row['query']))
        retrieved  = parse_docnos(row['retrieved_docs'])
        interacted = set(parse_interactions(row['interactions']))
        sessions[sid]['rel'].update(interacted)
        sessions[sid]['nonrel'].update(d for d in retrieved if d not in interacted)
    return {
        sid: {'queries':       d['queries'],
              'rel_docnos':    list(d['rel']),
              'nonrel_docnos': list(d['nonrel'])}
        for sid, d in sessions.items()
    }

# Testlauf
test_sessions = load_sessions(SNAPSHOT_FILES['snapshot-2.jsonl'])
sid0 = list(test_sessions.keys())[0]
print(f'Sessions snapshot-2: {len(test_sessions)}')
print(f'Session {sid0}: queries={test_sessions[sid0]["queries"]}')
print(f'  rel_docs={test_sessions[sid0]["rel_docnos"][:3]}')
print(f'  nonrel_docs={test_sessions[sid0]["nonrel_docnos"][:3]}')

Sessions snapshot-2: 119
Session 115: queries=['parenting style and identity of adolescents', "parent's identitystyle and identity of adolescents"]
  rel_docs=['82284189']
  nonrel_docs=['129020450', '9180748', '33178649']


In [6]:
def get_context(queries, mode):
    """Wählt Queries je nach query_mode aus (ohne letzte Query = die zu-vorhersagende)."""
    ctx = queries[:-1] if len(queries) > 1 else queries
    if mode == 'first': return ctx[0] if ctx else queries[0]
    if mode == 'last':  return ctx[-1] if ctx else queries[0]
    return '\n'.join(ctx)  # 'all'


def get_doc_text(docno):
    d = doc_lookup.get(str(docno), {})
    t = (d.get('title', '') or '').strip()
    a = (d.get('abstract', '') or '').strip()
    text = ('Title: ' + t + '\nAbstract: ' + a) if (t or a) else ('Document ID: ' + docno)
    return text[:RUN_CONFIG['max_doc_chars']]


# Topic-Cache: on-the-fly Generierung aus Session-Queries (Abgabe-Ansatz)
topic_cache = {}

def get_topic(sid, all_queries):
    """Generiert TREC-Topic on-the-fly aus den Session-Queries."""
    if sid in topic_cache:
        return topic_cache[sid]
    queries_text = '\n'.join(all_queries[:-1] if len(all_queries) > 1 else all_queries)
    prompt = (
        'You are an expert information retrieval analyst. '
        'Create a TREC-style search topic from these search queries. '
        'Return ONLY valid JSON with keys: title, description, narrative.\n\n'
        'Search queries: ' + queries_text
    )
    for attempt in range(3):
        try:
            resp = client.chat.completions.create(
                model=RUN_CONFIG['model'],
                messages=[{'role': 'user', 'content': prompt}],
                temperature=0.3, max_tokens=300,
            )
            raw = resp.choices[0].message.content.strip()
            match = re.search(r'\{.*\}', raw, re.DOTALL)
            if match:
                data = json.loads(match.group())
                t = str(data.get('title', queries_text[:30])).strip()
                d = str(data.get('description', '')).strip()
                n = str(data.get('narrative', '')).strip()
                if d and n:
                    topic = {'title': t, 'description': d, 'narrative': n, 'tfidf_terms': ''}
                    topic_cache[sid] = topic
                    return topic
        except Exception:
            time.sleep(2 ** attempt)
    fallback = {'title': all_queries[0][:30], 'description': 'Research about ' + all_queries[0],
                'narrative': 'N/A', 'tfidf_terms': ''}
    topic_cache[sid] = fallback
    return fallback


# ─── PROMPT A: Eigener Prompt ──────────────────────────────────────────────────
# doc_variant ändert die Instruktion:
#   'rel'         → Nutzer hat rel. Docs gefunden → tiefere/spezifischere Folgequeries
#   'non_rel'     → Nutzer hat nichts gefunden → alternative/umformulierte Queries
#   'contrastive' → neutrale Vorhersage

DOC_VARIANT_INSTRUCTION_A = {
    'rel': (
        'The researcher has already found some RELEVANT documents on this topic. '
        'Predict the 5 most likely next search queries that go DEEPER or MORE SPECIFIC '
        'into the topic — the researcher wants to explore further details or subtopics.'
    ),
    'non_rel': (
        'The researcher has NOT yet found useful documents — the retrieved results were NOT relevant. '
        'Predict the 5 most likely next search queries the researcher would use to '
        'REFORMULATE or try ALTERNATIVE angles to find what they are looking for.'
    ),
    'contrastive': (
        'Based on the research topic below, predict the 5 most likely next search queries '
        'that this researcher would naturally search for next.'
    ),
}

def build_prompt_A(sid, queries_text, all_queries, doc_variant='contrastive'):
    topic       = get_topic(sid, all_queries)
    tfidf       = topic.get('tfidf_terms', '').strip() or 'N/A'
    instruction = DOC_VARIANT_INSTRUCTION_A.get(doc_variant, DOC_VARIANT_INSTRUCTION_A['contrastive'])
    time.sleep(0.2)
    system = (
        'You are a precise research assistant. '
        'Your sole task is to generate follow-up search queries strictly based on the provided topic context. '
        'Rules:\n'
        '- Use ONLY information explicitly present in the context below.\n'
        '- Do NOT invent new topics, assumptions, or unrelated content.\n'
        '- Be specific, neutral, and concise.\n'
        '- Output ONLY valid JSON - no intro, no explanation, no commentary.'
    )
    user = (
        instruction + '\n\n'
        'TOPIC CONTEXT:\n'
        'Title:            ' + topic['title'] + '\n'
        'Description:      ' + topic['description'] + '\n'
        'Narrative:        ' + topic['narrative'] + '\n'
        'Previous Queries: ' + queries_text + '\n'
        'Key Terms:        ' + tfidf + '\n\n'
        'Output Format (strictly):\n'
        'Return ONLY valid JSON with key "queries": array of exactly 5 strings.\n'
        'No extra text, no markdown - only the JSON object.'
    )
    return system + '\n\n' + user


# ─── PROMPT B: Jüri-Prompt ────────────────────────────────────────────────────
# doc_variant steuert welche Docs im Prompt erscheinen

def build_prompt_B(queries_text, rel_docnos, nonrel_docnos, doc_variant):
    lines = [
        'Given some user queries, relevant and not relevant documents,',
        'predict the 5 most likely next search queries the user will type.',
        '',
        'Queries',
        'A person has typed these queries into a search engine:',
        queries_text,
        '',
    ]
    if doc_variant in ('rel', 'contrastive'):
        rel_block = ''
        for i, d in enumerate(rel_docnos[:RUN_CONFIG['n_rel_docs']], 1):
            rel_block += f'[Document {i}]\n' + get_doc_text(d) + '\n\n'
        lines += [
            '--- BEGIN RELEVANT DOCUMENTS CONTENT ---',
            rel_block.strip() or '(no clicked documents in this session)',
            '--- END RELEVANT DOCUMENTS CONTENT ---', '',
        ]
    if doc_variant in ('non_rel', 'contrastive'):
        nonrel_block = ''
        for i, d in enumerate(nonrel_docnos[:RUN_CONFIG['n_nonrel_docs']], 1):
            nonrel_block += f'[Document {i}]\n' + get_doc_text(d) + '\n\n'
        lines += [
            '--- BEGIN NOT RELEVANT DOCUMENTS CONTENT ---',
            nonrel_block.strip() or '(no skipped documents available)',
            '--- END NOT RELEVANT DOCUMENTS CONTENT ---', '',
        ]
    lines += [
        'Output Format and Structure:',
        'Return ONLY valid JSON with key "queries": array of exactly 5 strings.',
        'Rank from most to least likely. Keep queries concise and on-topic.',
        'No extra text, no markdown - only the JSON object.',
    ]
    return '\n'.join(lines)


print('Prompt-Funktionen definiert.')
print(f'Aktiver Modus: Prompt {RUN_CONFIG["prompt_variant"]} | doc_variant={RUN_CONFIG["doc_variant"]}')
if RUN_CONFIG['prompt_variant'] == 'A':
    print(f'  Instruktion ({RUN_CONFIG["doc_variant"]}): {DOC_VARIANT_INSTRUCTION_A[RUN_CONFIG["doc_variant"]][:90]}...')

Prompt-Funktionen definiert.
Aktiver Modus: Prompt A | doc_variant=non_rel
  Instruktion (non_rel): The researcher has NOT yet found useful documents — the retrieved results were NOT relevan...


In [7]:
def generate_next_queries(sid, sdata, retries=3):
    queries_text = get_context(sdata['queries'], RUN_CONFIG['query_mode'])

    if RUN_CONFIG['prompt_variant'] == 'A':
        prompt = build_prompt_A(sid, queries_text, sdata['queries'],
                                doc_variant=RUN_CONFIG['doc_variant'])
    else:
        prompt = build_prompt_B(queries_text, sdata['rel_docnos'],
                                sdata['nonrel_docnos'], RUN_CONFIG['doc_variant'])

    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=RUN_CONFIG['model'],
                messages=[{'role': 'user', 'content': prompt}],
                temperature=0.5, max_tokens=400,
            )
            raw = resp.choices[0].message.content.strip()
            match = re.search(r'\{.*\}', raw, re.DOTALL)
            if match:
                data = json.loads(match.group())
                qs = data.get('queries', [])
                if isinstance(qs, list) and len(qs) >= 1:
                    return [str(q).strip() for q in qs[:5]]
        except Exception as e:
            print(f'  Session {sid}, Attempt {attempt+1}: {e}')
            time.sleep(2 ** attempt)

    last = sdata['queries'][-2] if len(sdata['queries']) > 1 else sdata['queries'][0]
    return [last, last+' review', last+' study', last+' analysis', last+' overview']


# Testlauf
print('Testlauf...')
t = generate_next_queries(sid0, test_sessions[sid0])
for i, q in enumerate(t, 1):
    print(f'  {i}. {q}')

Testlauf...
  1. impact of authoritative parenting on adolescent self-concept
  2. effects of permissive parenting on teenage social behavior
  3. authoritarian parenting and identity formation in adolescents
  4. neglectful parenting influences on adolescent decision-making
  5. relationship between parenting styles and adolescent self-esteem


In [8]:
snapshot_results = {}

# ── Schritt 1: Snapshot-1 neu generieren ─────────────────────────────────────
snap1_name = 'snapshot-1.jsonl'
snap1_csv  = SNAPSHOT_FILES[snap1_name]
print(f'=== {snap1_name} (neu generieren mit {snap1_csv}) ===')
sessions = load_sessions(snap1_csv)
run1 = {'meta': {
    'team_name':   RUN_CONFIG['team_name'],
    'description': RUN_CONFIG['description'],
    'run_name':    RUN_CONFIG['run_name'],
    'query_mode':  RUN_CONFIG['query_mode'],
    'prompt':      RUN_CONFIG['prompt_variant'],
    'doc_variant': RUN_CONFIG['doc_variant'],
}}
for j, (sid, sdata) in enumerate(sessions.items(), 1):
    print(f'  [{j}/{len(sessions)}] {sdata["queries"][0][:60]}')
    run1[sid] = generate_next_queries(sid, sdata)
    time.sleep(0.3)
snapshot_results[snap1_name] = run1
print(f'  -> {len(sessions)} Sessions fertig.')

# ── Schritt 2: Snapshot-2 & -3 aus bestehendem ZIP laden ─────────────────────
existing_zip = Path(RUN_CONFIG['output_dir']) / f"task3_submission_{RUN_CONFIG['run_name']}.zip"

if existing_zip.exists():
    print(f'\nLade snapshot-2 & -3 aus: {existing_zip.name}')
    with zipfile.ZipFile(existing_zip, 'r') as zf:
        for snap_name in ['snapshot-2.jsonl', 'snapshot-3.jsonl']:
            if snap_name in zf.namelist():
                data = json.loads(zf.read(snap_name))
                sids = [k for k in data if k != 'meta']
                snapshot_results[snap_name] = data
                print(f'  {snap_name}: {len(sids)} Sessions übernommen ✓')
            else:
                print(f'  WARNUNG: {snap_name} nicht im ZIP gefunden!')
else:
    print('\nKein bestehender ZIP – generiere snapshot-2 & -3 ebenfalls neu...')
    for snap_name in ['snapshot-2.jsonl', 'snapshot-3.jsonl']:
        csv_path = SNAPSHOT_FILES[snap_name]
        print(f'\n=== {snap_name} ===')
        sessions = load_sessions(csv_path)
        run = {'meta': {
            'team_name':   RUN_CONFIG['team_name'],
            'description': RUN_CONFIG['description'],
            'run_name':    RUN_CONFIG['run_name'],
            'query_mode':  RUN_CONFIG['query_mode'],
            'prompt':      RUN_CONFIG['prompt_variant'],
            'doc_variant': RUN_CONFIG['doc_variant'],
        }}
        for j, (sid, sdata) in enumerate(sessions.items(), 1):
            print(f'  [{j}/{len(sessions)}] {sdata["queries"][0][:60]}')
            run[sid] = generate_next_queries(sid, sdata)
            time.sleep(0.3)
        snapshot_results[snap_name] = run
        print(f'  -> {len(sessions)} Sessions fertig.')

print('\nAlle Snapshots bereit!')
for k, v in snapshot_results.items():
    sids = [s for s in v if s != 'meta']
    print(f'  {k}: {len(sids)} Sessions')


In [ ]:
output_zip = os.path.join(
    RUN_CONFIG['output_dir'],
    'task3_submission_' + RUN_CONFIG['run_name'] + '.zip'
)

with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for snap_name, run_data in snapshot_results.items():
        zf.writestr(snap_name, json.dumps(run_data, ensure_ascii=False, indent=2))

errors = []
with zipfile.ZipFile(output_zip, 'r') as zf:
    print('Dateien im ZIP:', zf.namelist())
    for name in zf.namelist():
        data = json.loads(zf.read(name))
        sids = [k for k in data if k != 'meta']
        for key, val in data.items():
            if key == 'meta': continue
            if not isinstance(val, list) or len(val) < 1:
                errors.append(f'{name} Session {key}: keine Queries')
        print(f'  {name}: {len(sids)} Sessions')

if errors:
    print('FEHLER:', errors[:5])
else:
    counts = {k: len(v)-1 for k,v in snapshot_results.items()}
    print(f'\nSessions: {counts}')
    print(f'\nGespeichert: {output_zip}')
    print('Auf TIRA unter task-3-run-upload hochladen!')

Dateien im ZIP: ['snapshot-1.jsonl', 'snapshot-2.jsonl', 'snapshot-3.jsonl']
  snapshot-1.jsonl: 35 Sessions
  snapshot-2.jsonl: 119 Sessions
  snapshot-3.jsonl: 182 Sessions

Sessions: {'snapshot-1.jsonl': 35, 'snapshot-2.jsonl': 119, 'snapshot-3.jsonl': 182}

Gespeichert: runfiles_task3\task3_submission_openai_first_A_rel.zip
Auf TIRA unter task-3-run-upload hochladen!


---
## Alle 18 Runs – Config-Übersicht

Für jeden Run in **Cell 3** (`config`) die vier Werte ändern:

```python
# Run 1
query_mode='first', prompt_variant='A', doc_variant='rel',         run_name='openai_first_A_rel'
# Run 2
query_mode='first', prompt_variant='A', doc_variant='non_rel',     run_name='openai_first_A_non_rel'
# Run 3
query_mode='first', prompt_variant='A', doc_variant='contrastive', run_name='openai_first_A_contrastive'
# Run 4
query_mode='first', prompt_variant='B', doc_variant='rel',         run_name='openai_first_B_rel'
# Run 5
query_mode='first', prompt_variant='B', doc_variant='non_rel',     run_name='openai_first_B_non_rel'
# Run 6
query_mode='first', prompt_variant='B', doc_variant='contrastive', run_name='openai_first_B_contrastive'
# Run 7
query_mode='all',   prompt_variant='A', doc_variant='rel',         run_name='openai_all_A_rel'
# Run 8
query_mode='all',   prompt_variant='A', doc_variant='non_rel',     run_name='openai_all_A_non_rel'
# Run 9
query_mode='all',   prompt_variant='A', doc_variant='contrastive', run_name='openai_all_A_contrastive'
# Run 10
query_mode='all',   prompt_variant='B', doc_variant='rel',         run_name='openai_all_B_rel'
# Run 11
query_mode='all',   prompt_variant='B', doc_variant='non_rel',     run_name='openai_all_B_non_rel'
# Run 12
query_mode='all',   prompt_variant='B', doc_variant='contrastive', run_name='openai_all_B_contrastive'
# Run 13
query_mode='last',  prompt_variant='A', doc_variant='rel',         run_name='openai_last_A_rel'
# Run 14
query_mode='last',  prompt_variant='A', doc_variant='non_rel',     run_name='openai_last_A_non_rel'
# Run 15
query_mode='last',  prompt_variant='A', doc_variant='contrastive', run_name='openai_last_A_contrastive'
# Run 16
query_mode='last',  prompt_variant='B', doc_variant='rel',         run_name='openai_last_B_rel'
# Run 17
query_mode='last',  prompt_variant='B', doc_variant='non_rel',     run_name='openai_last_B_non_rel'
# Run 18
query_mode='last',  prompt_variant='B', doc_variant='contrastive', run_name='openai_last_B_contrastive'
```
